<a href="https://colab.research.google.com/github/PedroConst/Python-Bioprocessos/blob/main/Dia02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Dia 02: Python para Engenharia de Bioprocessos**
### Da análise de dados à simulação de crescimento microbiano

 Como podemos utilizar Python para analisar e modelar o crescimento de uma cultura microbiana?

### Revisão!

In [ ]:
import numpy as np

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("fermentacao_phb.csv")

df.head()

In [ ]:
tempo = df["Tempo (h)"].values
biomassa = df["Biomassa (g/L)"].values

In [ ]:
df_taxa = pd.DataFrame()

In [ ]:
dt = np.diff(tempo)

mu = np.log(biomassa[1:]/biomassa[:-1])/dt

df_taxa["Tempo médio (h)"] = (tempo[:-1]+tempo[1:])/2
df_taxa["Taxa específica (1/h)"] = mu


df_taxa.head()

## **Passo 1 – Organizando Cálculos com Funções em Python**

Até agora escrevemos códigos relativamente curtos. Entretanto, à medida que os problemas se tornam mais complexos, é comum precisarmos repetir os mesmos cálculos diversas vezes.

Em programação, uma boa prática consiste em organizar esses cálculos em **funções**. Uma função recebe informações de entrada, realiza uma sequência de operações e retorna um resultado.

Além de tornar o código mais organizado, as funções facilitam a reutilização de cálculos e reduzem a possibilidade de erros.

Nesta seção aprenderemos esse conceito utilizando exemplos relacionados ao nosso processo de produção de PHB.

### Um cálculo repetitivo

Suponha que desejemos calcular repetidamente a massa de biomassa presente em diferentes biorreatores.

Sabemos que:

$$
m=XV
$$

onde

- $X$ é a concentração celular (g/L);
- $V$ é o volume do reator (L).

Em vez de repetir esse cálculo diversas vezes, podemos criar uma função.

In [ ]:
def massa_biomassa(concentracao, volume):

    massa = concentracao * volume

    return massa

In [ ]:
massa = massa_biomassa(2.8, 3.5)

print(f"Massa = {massa:.2f} g")

In [ ]:
massa_biomassa(4.5,10)

In [ ]:
massa_biomassa(6.2,150)

In [ ]:
massa = massa_biomassa(
    df["Biomassa (g/L)"].iloc[-1],
    3.5
)

print(massa)

### Exemplo: rendimento biomassa/substrato

Uma grandeza frequentemente utilizada em Engenharia de Bioprocessos é o rendimento de biomassa em relação ao substrato consumido:

$$
Y_{X/S}=\frac{\Delta X}{-\Delta S}
$$

Vamos criar uma função para realizar esse cálculo.

In [ ]:
def rendimento(deltaX, deltaS):

    return deltaX/(-deltaS)

In [ ]:
deltaX = 2.4
deltaS = -6.8

YXS = rendimento(deltaX, deltaS)

print(YXS)

In [ ]:
deltaX = np.diff(df["Biomassa (g/L)"])

deltaS = np.diff(df["Substrato (g/L)"])

rendimento_exp = rendimento(deltaX, deltaS)

In [ ]:
rendimento_exp

### Exercício

Crie uma função chamada

```python
conversao(S0,S)
```

que calcule a conversão do substrato

$$
X=\frac{S_0-S}{S_0}
$$

Utilize essa função para calcular a conversão em todos os instantes da fermentação.

### Desafio

Crie uma função que receba um vetor contendo concentrações de biomassa e retorne:

- biomassa máxima;
- biomassa mínima;
- biomassa média.

Dica:

Utilize as funções do NumPy vistas anteriormente.

### Aplicação em Engenharia

À medida que um projeto evolui, os códigos utilizados para análise de dados e modelagem tornam-se cada vez maiores.

Organizar cálculos em funções torna os programas mais legíveis, facilita a identificação de erros e permite reutilizar rotinas em diferentes experimentos.

Nas próximas seções utilizaremos esse conceito para construir modelos matemáticos capazes de representar o crescimento microbiano durante a fermentação para produção de PHB.

## **Passo 2 – Construindo um Modelo para o Crescimento Microbiano**

Ao longo deste minicurso analisamos dados experimentais obtidos durante uma fermentação para produção de PHB.

Mas será que é possível ir além da análise dos dados?

Em Engenharia de Bioprocessos, frequentemente desejamos responder perguntas como:

- O que acontecerá se aumentarmos a concentração inicial de substrato?
- Quanto tempo será necessário para atingir uma determinada concentração celular?
- Como diferentes condições operacionais afetam o crescimento microbiano?

Responder essas perguntas realizando apenas experimentos pode ser caro e demorado. Por esse motivo, engenheiros utilizam **modelos matemáticos**, capazes de descrever o comportamento do sistema e realizar previsões antes mesmo da execução de novos experimentos.

Nesta seção conheceremos um dos modelos mais importantes da Engenharia de Bioprocessos: o modelo de Monod.



### O modelo mais simples possível

Suponha que cada célula presente no biorreator se divida continuamente à mesma velocidade.

Nesse caso, a velocidade de crescimento da biomassa será proporcional à quantidade de biomassa existente no sistema.

Esse comportamento pode ser representado por:

$$
\frac{dX}{dt} = μX
$$

onde

- $X$ é a concentração de biomassa;
- $\mu$ é a taxa específica de crescimento. Observe que essa é exatamente a grandeza calculada na seção anterior.


### 🔍 Interpretando o modelo

Esse modelo afirma que quanto maior a quantidade de biomassa presente no reator, maior será a velocidade de crescimento da cultura.

Embora simples, esse comportamento descreve razoavelmente bem a fase exponencial de muitos processos fermentativos.

Entretanto, ele assume que a taxa específica de crescimento $\mu$ permanece constante durante todo o experimento.

Será que essa hipótese é consistente com os dados experimentais analisados anteriormente?

In [ ]:
plt.figure(figsize=(7,4))

plt.plot(df_taxa["Tempo médio (h)"],
         df_taxa["Taxa específica (1/h)"],
         marker='o')

plt.xlabel("Tempo (h)")
plt.ylabel("μ (1/h)")
plt.grid()

plt.show()

Observando o gráfico acima, percebemos que a taxa específica não permanece constante.

Portanto, precisamos construir um modelo mais realista.

### O modelo de Monod

Uma das principais causas da redução da taxa de crescimento é a diminuição da disponibilidade de substrato ao longo da fermentação.

O modelo proposto por Jacques Monod descreve esse comportamento assumindo que a taxa específica depende da concentração de substrato disponível.

Nesse modelo,

$$
μ=μ_{max} \frac{S}{K_s+S}
$$

- $\mu_{max}$ representa a taxa específica máxima de crescimento;
- $S$ é a concentração de substrato;
- $K_s$ é a constante de saturação.

Quando existe muito substrato disponível, o crescimento aproxima-se de $\mu_{max}$.

À medida que o substrato é consumido, a taxa específica diminui.

Uma das vantagens de utilizar Python é que podemos transformar uma equação matemática em uma função reutilizável.

In [ ]:
def monod(S, mumax, Ks):
    return mumax*S/(Ks+S)

In [ ]:
S = np.linspace(0,25,200)

mu_max = 0.45
Ks = 3

mu = monod(S,mu_max,Ks)

In [ ]:
plt.figure(figsize=(7,4))

plt.plot(S,mu)

plt.xlabel("Substrato (g/L)")
plt.ylabel("μ (1/h)")

plt.grid()

plt.show()

Observe que o crescimento aumenta rapidamente para pequenas concentrações de substrato.

Entretanto, a partir de certo ponto, a curva tende a um valor máximo, indicando que o crescimento não pode aumentar indefinidamente, mesmo que exista grande quantidade de substrato disponível.

Esse comportamento é conhecido como **saturação** e é observado em diversos sistemas biológicos.

### Investigando os parâmetros do modelo

Uma das principais vantagens dos modelos matemáticos é permitir investigar o efeito de diferentes parâmetros sem a necessidade de realizar novos experimentos.

- O que acontece quando mudamos $\mu_{max}$?
- O que acontece quando mudamos $K_s$?

In [ ]:
plt.figure(figsize=(7,4))

for mu_max in [0.3,0.5,0.7]:

    Ks = 3.0

    mu = monod(S,mu_max,Ks)

    plt.plot(S,mu,label=f"μmax={mu_max}")

plt.xlabel("Substrato (g/L)")
plt.ylabel("μ (1/h)")

plt.legend()

plt.grid()

plt.show()

In [ ]:
plt.figure(figsize=(7,4))

for Ks in [1,3,6]:

    mu_max = 0.5

    mu = monod(S,mu_max,Ks)

    plt.plot(S,mu,label=f"Ks={Ks}")

plt.xlabel("Substrato (g/L)")
plt.ylabel("μ (1/h)")

plt.legend()

plt.grid()

plt.show()

Observe que dois parâmetros controlam o comportamento do modelo.

O parâmetro $\mu_{max}$ determina a velocidade máxima de crescimento que o microrganismo pode atingir.

Já $K_s$ controla a sensibilidade do crescimento à disponibilidade de substrato. Quanto menor seu valor, maior é a capacidade do microrganismo de crescer mesmo em baixas concentrações de substrato.

Esses parâmetros são característicos de cada sistema biológico e normalmente são obtidos a partir de dados experimentais.

### Exercício

Utilize a função `monod()` para calcular a taxa específica de crescimento considerando:

- $\mu_{max}=0.60\ \mathrm{h^{-1}}$
- $K_s=2.5\ \mathrm{g/L}$

Construa um gráfico de $\mu$ em função da concentração de substrato utilizando valores de \(S\) entre 0 e 30 g/L.

Em seguida, determine aproximadamente para qual concentração de substrato a taxa específica atinge metade de seu valor máximo.

### Desafio

Considere dois microrganismos utilizados para produção de PHB.

| Microrganismo | μmax (h⁻¹) | Ks (g/L) |
|--------------|------------|----------|
| A | 0.60 | 2.0 |
| B | 0.45 | 0.5 |

Utilizando a função `monod()`, compare os dois microrganismos em um mesmo gráfico.

- Qual deles apresenta maior vantagem quando a concentração de substrato é baixa?

- Qual deles apresenta maior velocidade de crescimento quando existe abundância de substrato?

### Aplicação em Engenharia

Modelos cinéticos como o de Monod são amplamente utilizados no projeto, operação e otimização de biorreatores. Eles permitem prever o comportamento de culturas microbianas sob diferentes condições de operação, reduzindo o número de experimentos necessários e auxiliando na tomada de decisões durante o desenvolvimento de processos biotecnológicos.

Nos próximos tópicos utilizaremos esse modelo para gerar um grande conjunto de dados sintéticos, simulando milhares de experimentos virtuais. Esses dados servirão como base para introduzir conceitos fundamentais de Machine Learning aplicados à Engenharia de Bioprocessos.

## **Passo 3 - Gerando Experimentos Virtuais com Modelos Matemáticos**

Até agora utilizamos o modelo de Monod para compreender como a disponibilidade de substrato influencia a taxa de crescimento microbiano.

Entretanto, modelos matemáticos possuem outra vantagem extremamente importante: eles podem ser utilizados para gerar novos dados.

Imagine que a equipe de engenharia deseja estudar centenas de combinações diferentes de microrganismos e condições operacionais. Realizar todos esses experimentos em laboratório demandaria semanas de trabalho e um elevado custo.

Como alternativa, podemos utilizar o modelo matemático para realizar **experimentos virtuais**, produzindo um grande conjunto de dados que poderá ser analisado posteriormente.

### Um experimento virtual

Vamos utilizar o modelo de Monod para simular como a taxa específica de crescimento varia em função da concentração de substrato para um determinado microrganismo.

In [ ]:
S = np.linspace(0, 25, 100)

mumax = 0.55
Ks = 2.5

mu = monod(S, mumax, Ks)

plt.figure(figsize=(7,4))

plt.plot(S, mu, linewidth=2)

plt.xlabel("Substrato (g/L)")
plt.ylabel("μ (1/h)")
plt.grid()

plt.show()

Acabamos de gerar um experimento completamente virtual.

Nenhum microrganismo foi cultivado em laboratório e nenhum reagente foi consumido. Mesmo assim, conseguimos obter informações sobre o comportamento esperado do processo utilizando apenas um modelo matemático.

Essa abordagem é amplamente utilizada em pesquisa e desenvolvimento para explorar diferentes cenários antes da realização de experimentos reais.

### Explorando diferentes condições operacionais

Na prática, um único experimento raramente é suficiente.

Vamos imaginar que a empresa deseja avaliar diversos microrganismos, cada um caracterizado por valores diferentes de $μ_{max}$ e $K_s$.

Podemos repetir automaticamente esse experimento diversas vezes.

In [ ]:
S = np.linspace(0,25,200)

plt.figure(figsize=(7,4))

for mumax, Ks in zip([0.40,0.50,0.60],
                     [1.5,3.0,5.0]):

    plt.plot(
        S,
        monod(S,mumax,Ks),
        label=f"μmax={mumax}, Ks={Ks}"
    )

plt.xlabel("Substrato (g/L)")
plt.ylabel("μ (1/h)")

plt.legend()

plt.grid()

plt.show()

Até este ponto ainda estamos escolhendo os parâmetros manualmente.

Mas e se quisermos estudar milhares de combinações diferentes?

Seria possível automatizar completamente esse processo?

### Construindo um banco de dados

Vamos gerar automaticamente milhares de combinações possíveis para os parâmetros do modelo.

Cada linha representará um experimento virtual diferente.

In [ ]:
np.random.seed(42)

N = 5000

S = np.random.uniform(0,25,N)

mumax = np.random.uniform(0.30,0.80,N)

Ks = np.random.uniform(0.5,8.0,N)

In [ ]:
mu = monod(S,mumax,Ks)

In [ ]:
df_ml = pd.DataFrame({

    "Substrato (g/L)":S,

    "μmax (1/h)":mumax,

    "Ks (g/L)":Ks,

    "μ (1/h)":mu

})

df_ml.head()

In [ ]:
df_ml.describe()

Em poucos milissegundos geramos um banco contendo cinco mil experimentos.

### 🔍 Interpretando os resultados
Observe que cada linha da tabela representa um experimento virtual diferente.

Embora esses dados não tenham sido obtidos em laboratório, eles foram produzidos por um modelo matemático baseado em princípios físicos.

Esse tipo de abordagem é bastante comum em aplicações modernas de engenharia, permitindo explorar rapidamente milhares de cenários antes da realização de experimentos reais.

### Exercício

Modifique o código para gerar um banco de dados contendo **10.000 experimentos virtuais**.

Em seguida:

- determine o maior valor de μ obtido;
- determine o menor valor de μ;
- calcule o valor médio da taxa específica de crescimento.

### Aplicação em Engenharia

A utilização de modelos matemáticos para gerar grandes conjuntos de dados é uma estratégia amplamente empregada em engenharia. Esses experimentos virtuais permitem avaliar rapidamente diferentes condições operacionais, reduzir custos experimentais e acelerar o desenvolvimento de novos processos.

Além disso, esses dados podem ser utilizados para treinar algoritmos de Machine Learning, construir modelos substitutos (*surrogate models*) e desenvolver gêmeos digitais (*digital twins*), tecnologias cada vez mais presentes na indústria.

## **Passo 4 - Construindo um Modelo Preditivo com Machine Learning**

Até este ponto do workshop utilizamos um modelo matemático para gerar milhares de experimentos virtuais.

Agora vamos inverter o problema.

Imagine que a equação de Monod foi perdida e restou apenas um grande conjunto de dados contendo milhares de experimentos.

Será que um computador seria capaz de aprender a relação entre essas variáveis apenas analisando os dados?

Essa é exatamente a ideia por trás do **Machine Learning**.

Em vez de programarmos explicitamente uma equação, fornecemos exemplos para que o algoritmo aprenda automaticamente os padrões presentes no conjunto de dados.

### Separando entradas e saídas

Em Machine Learning, normalmente dividimos o problema em:

- **Entradas (features):** informações utilizadas pelo algoritmo para realizar uma previsão;
- **Saída (target):** variável que desejamos prever.

Neste exemplo, utilizaremos como entradas:

- concentração de substrato;
- μmax;
- Ks.

Nosso objetivo será prever a taxa específica de crescimento μ.

In [ ]:
X = df_ml[["Substrato (g/L)",
           "μmax (1/h)",
           "Ks (g/L)"]]

y = df_ml["μ (1/h)"]

### Treinamento e teste

Antes de treinar um modelo é importante reservar uma parte dos dados para avaliar seu desempenho.

Uma prática comum consiste em utilizar aproximadamente 80% dos dados para treinamento e os 20% restantes para teste.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

O conjunto de treinamento será utilizado para ensinar o algoritmo.

Já o conjunto de teste contém dados que o modelo nunca viu anteriormente e serve para avaliar sua capacidade de realizar previsões em novas situações.

Essa etapa é fundamental para verificar se o modelo realmente aprendeu um padrão ou apenas memorizou os exemplos utilizados durante o treinamento.

### Regressão Linear

Nosso primeiro algoritmo será a Regressão Linear.

Embora simples, ela é uma das técnicas mais utilizadas para problemas de regressão e serve como ponto de partida para diversos estudos em Machine Learning.

In [ ]:
from sklearn.linear_model import LinearRegression

modelo = LinearRegression()

modelo.fit(X_train,y_train)

In [ ]:
y_pred = modelo.predict(X_test)

In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test,y_pred)

print(f"R² = {r2:.4f}")

In [ ]:
plt.figure(figsize=(6,6))

plt.scatter(y_test,y_pred,s=15)

plt.plot(
    [y_test.min(),y_test.max()],
    [y_test.min(),y_test.max()],
    '--r'
)

plt.xlabel("Valor verdadeiro")

plt.ylabel("Valor previsto")

plt.grid()

plt.show()

### 🔍 Interpretando os resultados

Quanto mais próximos os pontos estiverem da linha diagonal, melhor é a capacidade preditiva do modelo.

O coeficiente de determinação (**R²**) varia entre 0 e 1.

Valores próximos de 1 indicam que o modelo consegue reproduzir satisfatoriamente os dados utilizados na avaliação.

### Random Forest

Nem todos os problemas podem ser descritos adequadamente por uma relação linear.

Vamos comparar o desempenho da regressão linear com um algoritmo mais flexível: a **Random Forest**.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train,y_train)

y_pred_rf = rf.predict(X_test)

In [ ]:
r2_rf = r2_score(y_test,y_pred_rf)

print(f"R² = {r2_rf:.4f}")

In [ ]:
plt.figure(figsize=(6,6))

plt.scatter(
    y_test,
    y_pred_rf,
    s=15
)

plt.plot(
    [y_test.min(),y_test.max()],
    [y_test.min(),y_test.max()],
    '--r'
)

plt.xlabel("Valor verdadeiro")

plt.ylabel("Valor previsto")

plt.grid()

plt.show()

In [ ]:
print(f"Regressão Linear : R²= {r2:.4f}")

print(f"Random Forest : R²= {r2_rf:.4f}")

### 🔍 Interpretando os resultados

Ambos os algoritmos conseguem aprender relações presentes nos dados, porém modelos mais flexíveis tendem a representar melhor comportamentos não lineares.

Neste exemplo, como os dados foram gerados por um modelo matemático não linear, espera-se que a Random Forest apresente desempenho superior ao da Regressão Linear.

### Exercício

Modifique o tamanho do conjunto de teste para:

- 10%
- 30%
- 40%

O valor de R² muda significativamente?

Discuta com seus colegas por que diferentes divisões dos dados podem influenciar a avaliação do modelo.

### Desafio

Treine um novo modelo de Machine Learning utilizando apenas duas variáveis de entrada:

- concentração de substrato;
- μmax.

Compare o desempenho obtido com aquele alcançado utilizando as três variáveis.

Qual informação importante foi perdida ao remover o parâmetro Ks?

### Aplicação em Engenharia

Técnicas de Machine Learning são utilizadas em diversas áreas da Engenharia de Bioprocessos, incluindo monitoramento de fermentações, detecção de falhas, previsão de produtividade, controle avançado de processos e otimização operacional.

Na prática, modelos baseados em dados não substituem necessariamente os modelos fenomenológicos. Em muitas aplicações modernas, ambas as abordagens são utilizadas de forma complementar: modelos matemáticos incorporam conhecimento físico do processo, enquanto algoritmos de Machine Learning exploram padrões presentes nos dados experimentais para aumentar a capacidade preditiva dos sistemas.

# Conclusão

## Voltando ao desafio inicial

No início deste minicurso, assumimos o papel de engenheiros recém-contratados por uma empresa de biotecnologia interessada em desenvolver um processo sustentável para produção de polihidroxibutirato (PHB) utilizando melaço de cana-de-açúcar como matéria-prima.

O objetivo da equipe era utilizar ferramentas computacionais para compreender melhor o processo fermentativo, reduzir o número de experimentos necessários e apoiar o desenvolvimento de uma tecnologia mais eficiente e economicamente viável.

Ao longo dos dois dias do workshop, percorremos exatamente esse caminho.

Primeiro, aprendemos a importar, organizar e visualizar dados experimentais obtidos durante a fermentação. Em seguida, utilizamos esses dados para calcular taxas de crescimento e compreender o comportamento do cultivo.

Depois, introduzimos o modelo de Monod para representar matematicamente a relação entre a disponibilidade de substrato e a velocidade de crescimento microbiano. A partir desse modelo, construímos milhares de experimentos virtuais, permitindo explorar diferentes condições operacionais sem a necessidade de realizar novos ensaios em laboratório.

Por fim, utilizamos esse conjunto de dados para treinar um modelo de Machine Learning capaz de aprender padrões presentes no processo e realizar previsões sobre o comportamento do sistema.

Embora o exemplo apresentado neste minicurso tenha sido bastante simplificado, ele reproduz o fluxo de trabalho utilizado atualmente em diversos projetos de pesquisa e desenvolvimento na indústria de bioprocessos.

Hoje, ferramentas computacionais como Python, modelagem matemática e Machine Learning desempenham um papel cada vez mais importante no desenvolvimento de bioprocessos sustentáveis, contribuindo para reduzir custos experimentais, acelerar a inovação e apoiar decisões baseadas em dados.

Esperamos que este minicurso tenha mostrado que a programação não é apenas uma habilidade computacional, mas uma ferramenta poderosa para compreender, modelar e resolver problemas reais de Engenharia de Bioprocessos.

## O caminho percorrido

Durante este minicurso seguimos um fluxo semelhante ao utilizado por engenheiros e pesquisadores no desenvolvimento de processos biotecnológicos modernos.


```
Experimentos reais
        │
        ▼
 Análise de dados
        │
        ▼
 Modelo matemático
        │
        ▼
 Experimentos virtuais
        │
        ▼
 Machine Learning
        │
        ▼
 Predição e tomada de decisão
        │
        ▼
 Melhoria do processo
```
